# Whitecaps FC — Player Scouting & Career Projection Model

This notebook implements the scouting analytics pipeline used by the Whitecaps performance science team:

1. **Scouting Career Score** — CatBoost model that rates player value based on on-field metrics
2. **Aging Curve Model** — Scipy curve fitting to project career trajectories by position
3. **Cortex AI Scouting Reports** — LLM-generated narrative reports for technical staff
4. **Academy Pipeline Integration** — Scoring academy prospects against first-team benchmarks

## 1. Load Player Data

In [ ]:
%%sql -r player_stats
SELECT
    PLAYER_ID,
    PLAYER_NAME,
    POSITION,
    SQUAD,
    NATIONALITY,
    AGE,
    APPEARANCES,
    GOALS,
    ASSISTS,
    MINUTES_PLAYED,
    PASS_COMPLETION_PCT,
    TACKLES_PER_90,
    XG,
    XA,
    CONTRACT_END_DATE,
    MARKET_VALUE_USD,
    IS_DESIGNATED_PLAYER,
    IS_U22_INITIATIVE,
    IS_HOMEGROWN
FROM WHITECAPS_DEMO_DB.GOLD_ANALYTICS.PLAYER_STATS
ORDER BY PLAYER_ID

In [ ]:
%%sql -r academy_players
SELECT
    PLAYER_ID,
    PLAYER_NAME,
    AGE,
    POSITION,
    ACADEMY_LEVEL,
    HEIGHT_CM,
    WEIGHT_KG,
    TRAINING_ATTENDANCE_PCT,
    GPA,
    SCHOLARSHIP_STATUS,
    NOTES
FROM WHITECAPS_DEMO_DB.GOLD_ANALYTICS.ACADEMY_PLAYERS
WHERE IS_MINOR = FALSE
ORDER BY PLAYER_ID

## 2. Scouting Career Score — CatBoost Model

Mirrors the production `SCOUTING_CAREER_SCORE` UDF running on the Whitecaps account.
Builds a composite scouting score from on-field performance features using gradient boosting.

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

df = player_stats.to_pandas()

# Feature engineering — composite target mirrors the production scoring logic
df['GOAL_CONTRIBUTION'] = df['GOALS'] + df['ASSISTS']
df['MINUTES_PER_APPEARANCE'] = np.where(df['APPEARANCES'] > 0, df['MINUTES_PLAYED'] / df['APPEARANCES'], 0)
df['XG_OVERPERFORMANCE'] = df['GOALS'] - df['XG']
df['XA_OVERPERFORMANCE'] = df['ASSISTS'] - df['XA']

# Scouting score target — weighted composite (same formula as production SCOUTING_CAREER_SCORE)
df['SCOUTING_SCORE'] = (
    0.20 * (df['GOAL_CONTRIBUTION'] / df['GOAL_CONTRIBUTION'].max()) +
    0.15 * (df['XG_OVERPERFORMANCE'] / df['XG_OVERPERFORMANCE'].abs().max()) +
    0.15 * (df['XA_OVERPERFORMANCE'] / df['XA_OVERPERFORMANCE'].abs().max()) +
    0.15 * (df['PASS_COMPLETION_PCT'] / 100) +
    0.10 * (df['TACKLES_PER_90'] / df['TACKLES_PER_90'].max()) +
    0.10 * (df['MINUTES_PER_APPEARANCE'] / 90) +
    0.10 * np.where(df['AGE'] <= 24, 1.0, np.where(df['AGE'] <= 28, 0.8, 0.5)) +
    0.05 * df['IS_HOMEGROWN'].astype(int)
) * 100

# Encode position for model
le = LabelEncoder()
df['POSITION_ENC'] = le.fit_transform(df['POSITION'])

features = ['AGE', 'APPEARANCES', 'GOALS', 'ASSISTS', 'MINUTES_PLAYED',
            'PASS_COMPLETION_PCT', 'TACKLES_PER_90', 'XG', 'XA',
            'POSITION_ENC', 'GOAL_CONTRIBUTION', 'MINUTES_PER_APPEARANCE',
            'XG_OVERPERFORMANCE', 'XA_OVERPERFORMANCE']

X = df[features]
y = df['SCOUTING_SCORE']

# Train CatBoost model
model = CatBoostRegressor(
    iterations=200,
    depth=4,
    learning_rate=0.1,
    loss_function='RMSE',
    verbose=0
)
model.fit(X, y)

# Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print(f'Model R² (5-fold CV): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

# Predict and rank
df['PREDICTED_SCORE'] = model.predict(X)
df['RANK'] = df['PREDICTED_SCORE'].rank(ascending=False).astype(int)

print(f'\nTop 10 players by scouting score:')
print(df[['RANK', 'PLAYER_NAME', 'POSITION', 'AGE', 'PREDICTED_SCORE']]
      .sort_values('RANK').head(10).to_string(index=False))

### Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance = model.get_feature_importance()
feat_imp = pd.DataFrame({'Feature': features, 'Importance': importance})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_imp['Feature'], feat_imp['Importance'], color='#00205B')
ax.set_xlabel('Importance')
ax.set_title('Scouting Career Score — Feature Importance (CatBoost)')
plt.tight_layout()
plt.show()

## 3. Aging Curve Model — Career Trajectory Projection

Mirrors the production `SCOUTING_CAREER_FIT_AGING` procedure.
Fits a parametric aging curve (quadratic peak model) to project when players hit peak value and how their output declines by position.

In [ ]:
from scipy.optimize import curve_fit
from scipy.stats import norm

def aging_curve(age, peak_age, peak_val, decay_rate):
    """Quadratic aging model — symmetric rise/decline around peak."""
    return peak_val * np.exp(-decay_rate * (age - peak_age) ** 2)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
positions = ['Forward', 'Midfielder', 'Defender']
colors = ['#C8102E', '#00205B', '#5C6670']

aging_results = []

for idx, pos in enumerate(positions):
    pos_df = df[df['POSITION'] == pos].sort_values('AGE')
    if len(pos_df) < 3:
        continue

    ages = pos_df['AGE'].values.astype(float)
    scores = pos_df['PREDICTED_SCORE'].values

    try:
        popt, _ = curve_fit(aging_curve, ages, scores,
                           p0=[27, scores.max(), 0.01],
                           bounds=([18, 0, 0.001], [35, 200, 0.5]),
                           maxfev=5000)
        peak_age, peak_val, decay = popt
    except RuntimeError:
        peak_age, peak_val, decay = 27.0, scores.mean(), 0.02
        popt = [peak_age, peak_val, decay]

    aging_results.append({'Position': pos, 'Peak Age': round(peak_age, 1),
                          'Peak Score': round(peak_val, 1), 'Decay Rate': round(decay, 4)})

    age_range = np.linspace(18, 38, 100)
    projected = aging_curve(age_range, *popt)

    ax = axes[idx]
    ax.scatter(ages, scores, color=colors[idx], s=80, zorder=5, label='Actual')
    ax.plot(age_range, projected, color=colors[idx], linewidth=2, linestyle='--', label='Projected')
    ax.axvline(x=peak_age, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(f'{pos}s — Peak at {peak_age:.1f}y', fontsize=12)
    ax.set_xlabel('Age')
    ax.set_ylabel('Scouting Score')
    ax.legend(fontsize=9)
    ax.set_xlim(18, 38)

plt.suptitle('Whitecaps FC — Career Aging Curves by Position', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

aging_df = pd.DataFrame(aging_results)
print(aging_df.to_string(index=False))

## 4. Write Scouting Scores Back to Snowflake

Materialize the model outputs so the Club Agent and BI dashboards can serve them.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

output_df = df[['PLAYER_ID', 'PLAYER_NAME', 'POSITION', 'AGE',
                'PREDICTED_SCORE', 'RANK', 'XG_OVERPERFORMANCE',
                'XA_OVERPERFORMANCE', 'GOAL_CONTRIBUTION']].copy()
output_df.columns = [c.upper() for c in output_df.columns]

sp_df = session.create_dataframe(output_df)
sp_df.write.mode('overwrite').save_as_table(
    'WHITECAPS_DEMO_DB.GOLD_ANALYTICS.SCOUTING_CAREER_SCORES'
)
print(f'Wrote {len(output_df)} player scores to SCOUTING_CAREER_SCORES')

## 5. Cortex AI — Scouting Report Generation

Uses Snowflake Cortex `AI_COMPLETE` to generate narrative scouting reports — the same pattern that feeds the Whitecaps Club Agent's natural-language responses.

In [ ]:
%%sql -r scouting_reports
SELECT
    s.PLAYER_NAME,
    s.POSITION,
    s.AGE,
    s.PREDICTED_SCORE,
    s.RANK,
    p.MARKET_VALUE_USD,
    p.CONTRACT_END_DATE,
    p.IS_DESIGNATED_PLAYER,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-6',
        CONCAT(
            'You are a professional soccer scout for Vancouver Whitecaps FC (MLS). ',
            'Write a concise 3-sentence scouting assessment for this player. ',
            'Include their strengths, areas to watch, and contract recommendation. ',
            'Player: ', s.PLAYER_NAME,
            ', Position: ', s.POSITION,
            ', Age: ', s.AGE::VARCHAR,
            ', Scouting Score: ', ROUND(s.PREDICTED_SCORE, 1)::VARCHAR, '/100',
            ', Goals+Assists: ', s.GOAL_CONTRIBUTION::VARCHAR,
            ', xG Overperformance: ', ROUND(s.XG_OVERPERFORMANCE, 1)::VARCHAR,
            ', Market Value: $', s.MARKET_VALUE_USD::VARCHAR,
            ', Contract Ends: ', p.CONTRACT_END_DATE::VARCHAR,
            ', Designated Player: ', p.IS_DESIGNATED_PLAYER::VARCHAR
        )
    ) AS AI_SCOUTING_REPORT
FROM WHITECAPS_DEMO_DB.GOLD_ANALYTICS.SCOUTING_CAREER_SCORES s
JOIN WHITECAPS_DEMO_DB.GOLD_ANALYTICS.PLAYER_STATS p USING (PLAYER_ID)
WHERE s.RANK <= 5
ORDER BY s.RANK

In [ ]:
reports_df = scouting_reports.to_pandas()

for _, row in reports_df.iterrows():
    print(f"{'='*60}")
    print(f"#{int(row['RANK'])} — {row['PLAYER_NAME']} ({row['POSITION']}, age {row['AGE']})")
    print(f"Score: {row['PREDICTED_SCORE']:.1f}/100 | Value: ${row['MARKET_VALUE_USD']:,.0f}")
    print(f"Contract: {row['CONTRACT_END_DATE']} | DP: {row['IS_DESIGNATED_PLAYER']}")
    print(f"\n{row['AI_SCOUTING_REPORT']}")
    print()

## 6. Academy Prospect Scoring

Score academy players against first-team benchmarks to identify promotion candidates.

In [ ]:
%%sql -r academy_vs_first_team
WITH first_team_benchmarks AS (
    SELECT
        POSITION,
        AVG(PASS_COMPLETION_PCT) AS AVG_PASS_PCT,
        AVG(TACKLES_PER_90) AS AVG_TACKLES,
        AVG(GOALS + ASSISTS) AS AVG_CONTRIBUTIONS,
        AVG(MARKET_VALUE_USD) AS AVG_VALUE
    FROM WHITECAPS_DEMO_DB.GOLD_ANALYTICS.PLAYER_STATS
    WHERE SQUAD = 'MLS First Team'
    GROUP BY POSITION
)
SELECT
    a.PLAYER_NAME,
    a.AGE,
    a.POSITION,
    a.ACADEMY_LEVEL,
    a.TRAINING_ATTENDANCE_PCT,
    a.GPA,
    a.SCHOLARSHIP_STATUS,
    b.AVG_PASS_PCT AS FIRST_TEAM_AVG_PASS_PCT,
    b.AVG_CONTRIBUTIONS AS FIRST_TEAM_AVG_CONTRIBUTIONS,
    a.NOTES
FROM WHITECAPS_DEMO_DB.GOLD_ANALYTICS.ACADEMY_PLAYERS a
LEFT JOIN first_team_benchmarks b ON a.POSITION = b.POSITION
WHERE a.IS_MINOR = FALSE
ORDER BY a.TRAINING_ATTENDANCE_PCT DESC

In [ ]:
acad_df = academy_vs_first_team.to_pandas()

# Academy readiness score — training attendance + GPA + scholarship tier
scholarship_weights = {'Homegrown Pathway': 1.0, 'Full Scholarship': 0.9,
                       'Development Contract': 0.7, 'Partial Scholarship': 0.5, 'Trial': 0.3}
acad_df['SCHOLARSHIP_WEIGHT'] = acad_df['SCHOLARSHIP_STATUS'].map(scholarship_weights).fillna(0.3)
acad_df['READINESS_SCORE'] = (
    0.50 * (acad_df['TRAINING_ATTENDANCE_PCT'] / 100) +
    0.25 * (acad_df['GPA'] / 4.0) +
    0.25 * acad_df['SCHOLARSHIP_WEIGHT']
) * 100

fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {'Forward': '#C8102E', 'Midfielder': '#00205B', 'Defender': '#5C6670', 'Goalkeeper': '#4CAF50'}
bar_colors = [colors_map.get(p, '#999') for p in acad_df['POSITION']]
bars = ax.barh(acad_df['PLAYER_NAME'], acad_df['READINESS_SCORE'], color=bar_colors)
ax.axvline(x=75, color='red', linestyle='--', label='First-team ready threshold')
ax.set_xlabel('Readiness Score')
ax.set_title('Academy Players — First-Team Readiness Assessment')
ax.legend()
plt.tight_layout()
plt.show()

print('\nPromotion candidates (score >= 75):')
candidates = acad_df[acad_df['READINESS_SCORE'] >= 75]
print(candidates[['PLAYER_NAME', 'POSITION', 'AGE', 'READINESS_SCORE', 'SCHOLARSHIP_STATUS']].to_string(index=False))

## 7. Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Score distribution
axes[0, 0].hist(df['PREDICTED_SCORE'], bins=12, color='#00205B', edgecolor='white')
axes[0, 0].set_title('Scouting Score Distribution')
axes[0, 0].set_xlabel('Score')
axes[0, 0].set_ylabel('Players')

# 2. Age vs Score scatter
for pos in positions:
    pdata = df[df['POSITION'] == pos]
    axes[0, 1].scatter(pdata['AGE'], pdata['PREDICTED_SCORE'],
                       label=pos, s=80, alpha=0.8)
axes[0, 1].set_title('Age vs Scouting Score')
axes[0, 1].set_xlabel('Age')
axes[0, 1].set_ylabel('Score')
axes[0, 1].legend()

# 3. Market value vs score
axes[1, 0].scatter(df['MARKET_VALUE_USD'] / 1e6, df['PREDICTED_SCORE'],
                   c='#C8102E', s=80, alpha=0.7)
for _, row in df.iterrows():
    if row['RANK'] <= 5:
        axes[1, 0].annotate(row['PLAYER_NAME'],
                            (row['MARKET_VALUE_USD']/1e6, row['PREDICTED_SCORE']),
                            fontsize=8, ha='left')
axes[1, 0].set_title('Market Value vs Scouting Score')
axes[1, 0].set_xlabel('Market Value ($M)')
axes[1, 0].set_ylabel('Score')

# 4. Position averages
pos_avg = df.groupby('POSITION')['PREDICTED_SCORE'].mean().sort_values()
axes[1, 1].barh(pos_avg.index, pos_avg.values, color=['#5C6670', '#00205B', '#C8102E', '#4CAF50'][:len(pos_avg)])
axes[1, 1].set_title('Average Score by Position')
axes[1, 1].set_xlabel('Avg Scouting Score')

plt.suptitle('Whitecaps FC — Scouting Analytics Dashboard', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()